In [2]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)
documents = [file.parse() for file in reader.read()]

In [3]:
data_gen_instructions = """
You emulate a student who is taking our LLM course.
You are given one lesson page from the course.
Formulate 5 questions this student might ask that are answered by this page.

Rules:
- The page should contain the answer to each question.
- Make the questions complete and not too short.
- Use as few words as possible from the page; don't copy its phrasing.
- The questions should resemble how people actually ask things online:
  not too formal, not too short, not too long.
- Ask about the content of the lesson, not about its formatting or filename.
""".strip()

In [4]:
documents[0]

{'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour phone uses a simp

In [5]:
documents_llm = []

for doc in documents:
    if (doc["filename"] == "01-agentic-rag/lessons/01-intro.md"
        or doc["filename"] == "01-agentic-rag/lessons/02-environment.md"
        or doc["filename"] == "01-agentic-rag/lessons/03-rag.md"):
        documents_llm.append(doc)

len(documents_llm)

3

In [6]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
client = OpenAI()

In [7]:
from pydantic import BaseModel


class QuestionsList(BaseModel):
    questions: list[str]

In [8]:
from tqdm.auto import tqdm

from evaluation_utils import llm_structured_retry

results = []

for doc in tqdm(documents_llm):
    questions, usage = llm_structured_retry(
        client,
        data_gen_instructions,
        doc["content"],
        QuestionsList,
    )

    print(doc["filename"], "input_tokens:", usage.input_tokens)

    results.append({
        "filename": doc["filename"],
        "questions": questions.questions,
        "input_tokens": usage.input_tokens,
    })

  0%|          | 0/3 [00:00<?, ?it/s]

01-agentic-rag/lessons/01-intro.md input_tokens: 943
01-agentic-rag/lessons/02-environment.md input_tokens: 1157
01-agentic-rag/lessons/03-rag.md input_tokens: 1569


In [ ]:
avg_input_tokens = sum(r["input_tokens"] for r in results) / len(results)
avg_input_tokens

In [9]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

In [13]:
from minsearch import Index

index = Index(text_fields=["content"], keyword_fields=["filename"])
index.fit(chunks)

In [18]:
from embedder import Embedder

embedder = Embedder()

contents = [item['content'] for item in chunks]
encoded = embedder.encode_batch(contents)

# Add encoding back to each item
for item, encoding in zip(chunks, encoded):
    item['encoding'] = encoding
    # Now you have: item['content'], item['filename'], item['encoding']

import numpy as np
X = np.array(encoded)

In [ ]:
from minsearch import VectorSearch

vindex = VectorSearch(keyword_fields=["filename"])
vindex.fit(X, chunks)

In [19]:
def text_search(query, num_results=5):
    return index.search(query, num_results=num_results)


def vector_search(query, num_results=5):
    query_vector = embedder.encode(query)
    return vindex.search(query_vector, num_results=num_results)

In [ ]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

In [ ]:
def hybrid_search(query, k=60):
    text_results = text_search(query, num_results=10)
    vector_results = vector_search(query, num_results=10)
    return rrf([text_results, vector_results], k=k)

In [16]:
import csv

with open("ground-truth.csv") as f:
    ground_truth = list(csv.DictReader(f))

len(ground_truth)

360

In [17]:
q = ground_truth[0]["question"]
q

"What exactly is a retrieval-augmented generation system, and why does it help with answers that the model wouldn't know on its own?"

In [21]:
text_results = text_search(q, num_results=5)
text_results

[{'start': 3000,
  'content': 'we drop it.\n\nBuild a prompt that includes both the question and the context:\n\n```python\nprompt = f"""\nYour task is to answer questions from the course participants\nbased on the provided context.\n\nUse the context to find relevant information and provide accurate\nanswers. If the answer is not found in the context,\nrespond with "I don\'t know."\n\nQuestion:\n{question}\n\nContext:\n{context}\n"""\n```\n\nInstead of sending the raw question to the LLM, we send this prompt:\n\n```python\nanswer = llm(prompt)\nprint(answer)\n```\n\nAfter that, the answer is correct: "Yes, you can still join. If you want to\nreceive a certificate, you need to submit your project while\nsubmissions are still open."\n\nThis is the answer we actually want to give to our students. What we\njust did is nothing but RAG.\n\n## Retrieval plus generation\n\nRAG stands for Retrieval-Augmented Generation. Generation is the LLM\nproducing text, and retrieval is search. We retriev

In [22]:
vector_results = vector_search(q, num_results=5)
vector_results

[{'start': 0,
  'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour ph

In [54]:
def compute_relevance(q, search_function):
    doc_id = q["filename"]
    results = search_function(query=q["question"])

    relevance = []
    for d in results:
        relevance.append(int(d["filename"] == doc_id))

    return relevance

In [55]:
def compute_relevance_total(ground_truth, search_function):
    relevance_total = []

    for q in tqdm(ground_truth):
        relevance = compute_relevance(q, search_function)
        relevance_total.append(relevance)

    return relevance_total

In [56]:
def hit_rate(relevance):
    cnt = 0

    for line in relevance:
        if 1 in line:
            cnt = cnt + 1

    return cnt / len(relevance)

In [57]:
def mrr(relevance):
    total_score = 0.0

    for line in relevance:
        for rank in range(len(line)):
            if line[rank] == 1:
                score = 1 / (rank + 1)
                total_score = total_score + score
                break

    return total_score / len(relevance)
    

In [58]:
def evaluate(ground_truth, search_function):
    relevance_total = compute_relevance_total(ground_truth, search_function)

    return {
        "hit_rate": hit_rate(relevance_total),
        "mrr": mrr(relevance_total),
    }

In [59]:
evaluate(ground_truth, text_search)

  0%|          | 0/360 [00:00<?, ?it/s]

{'hit_rate': 0.7583333333333333, 'mrr': 0.5942592592592594}

In [60]:
evaluate(ground_truth, vector_search)

  0%|          | 0/360 [00:00<?, ?it/s]

{'hit_rate': 0.725, 'mrr': 0.5486111111111112}

In [61]:
from functools import partial

hybrid_results = {}

for k in [1, 50, 100, 200]:
    search_function = partial(hybrid_search, k=k)
    hybrid_results[k] = evaluate(ground_truth, search_function)

hybrid_results

  0%|          | 0/360 [00:00<?, ?it/s]

  0%|          | 0/360 [00:00<?, ?it/s]

  0%|          | 0/360 [00:00<?, ?it/s]

  0%|          | 0/360 [00:00<?, ?it/s]

{1: {'hit_rate': 0.8388888888888889, 'mrr': 0.6481944444444449},
 50: {'hit_rate': 0.8361111111111111, 'mrr': 0.637916666666667},
 100: {'hit_rate': 0.8361111111111111, 'mrr': 0.637916666666667},
 200: {'hit_rate': 0.8361111111111111, 'mrr': 0.637916666666667}}

In [62]:
best_k = max(hybrid_results, key=lambda k: hybrid_results[k]["mrr"])
best_k, hybrid_results[best_k]["mrr"]

(1, 0.6481944444444449)